In [0]:
%run /Workspace/Users/ceresrain@yahoo.com/Butterfly_effect/short-form-butterfly/config/00_config/config


Config loaded.
Bronze: bootcamp_students.tiffani_ceresrain_bronze
Silver: bootcamp_students.tiffani_ceresrain_silver
Gold:   bootcamp_students.tiffani_ceresrain_gold
Films loaded: 50


In [0]:
# — Silver | Clean and normalize Google Trends data
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load Bronze table
df_trends = spark.table(f"{BRONZE_PATH}.google_trends_raw")

# Convert date column to proper date type
df_trends = df_trends.withColumn("date", F.to_date(F.col("date")))

# Calculate days before release for each row
df_trends = df_trends.withColumn(
    "days_before_release",
    F.datediff(F.col("release_date").cast("date"), F.col("date"))
)

# Per film window — for normalization
film_window = Window.partitionBy("film")

# Key metrics per film
df_trends_silver = (df_trends
    # Peak hype score in the 90-day window
    .withColumn("hype_peak_score", 
        F.max("interest_score").over(film_window))
    # Average score across full window
    .withColumn("hype_avg_score",
        F.avg("interest_score").over(film_window))
    # When did peak occur (days before release)
    .withColumn("peak_interest",
        F.max("interest_score").over(film_window))
    # Assign era
   .withColumn("era", F.when(
        F.year(F.col("release_date").cast("date")) <= 2014, 
        "Pre Short-Form (2005-2014)")
        .otherwise("Short-Form Era (2015-2025)"))
)

# Collapse to one row per film (film-level summary)
df_trends_agg = (df_trends_silver
    .groupBy("film", "release_date", "genre", "era",
              "hype_peak_score", "hype_avg_score")
    .agg(
        F.count("*").alias("days_tracked"),
        F.sum(F.when(F.col("interest_score") > 50, 1)
              .otherwise(0)).alias("days_above_50")
    )
)

print(f"✅ Trends silver ready — {df_trends_agg.count()} films")
display(df_trends_agg)

✅ Trends silver ready — 50 films


film,release_date,genre,era,hype_peak_score,hype_avg_score,days_tracked,days_above_50
A Quiet Place,2018-04-06,Horror,Short-Form Era (2015-2025),100,5.956043956043956,91,2
Alien: Romulus,2024-08-16,Horror,Short-Form Era (2015-2025),100,5.967032967032967,91,2
Avatar,2009-12-18,Sci-Fi,Pre Short-Form (2005-2014),100,13.252747252747254,91,1
Barbie,2023-07-21,Comedy,Short-Form Era (2015-2025),100,9.0,91,2
Batman Begins,2005-06-15,Action,Pre Short-Form (2005-2014),100,11.846153846153847,91,2
Black Panther,2018-02-16,Action,Short-Form Era (2015-2025),100,6.670329670329671,91,1
Bohemian Rhapsody,2018-11-02,Drama,Short-Form Era (2015-2025),100,7.78021978021978,91,1
Bridesmaids,2011-05-13,Comedy,Pre Short-Form (2005-2014),100,10.780219780219781,91,1
Brokeback Mountain,2005-12-09,Drama,Pre Short-Form (2005-2014),100,12.87912087912088,91,1
Casino Royale,2006-11-17,Action,Pre Short-Form (2005-2014),100,9.516483516483516,91,1


In [0]:
# — Save to Silver Delta table
(df_trends_agg
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{SILVER_PATH}.trends_silver")
)

print(f"✅ Saved to {SILVER_PATH}.trends_silver")
print(f"   Rows written: {df_trends_agg.count()}")

display(spark.table(f"{SILVER_PATH}.trends_silver"))

✅ Saved to bootcamp_students.tiffani_ceresrain_silver.trends_silver
   Rows written: 50


film,release_date,genre,era,hype_peak_score,hype_avg_score,days_tracked,days_above_50
A Quiet Place,2018-04-06,Horror,Short-Form Era (2015-2025),100,5.956043956043956,91,2
Alien: Romulus,2024-08-16,Horror,Short-Form Era (2015-2025),100,5.967032967032967,91,2
Avatar,2009-12-18,Sci-Fi,Pre Short-Form (2005-2014),100,13.252747252747254,91,1
Barbie,2023-07-21,Comedy,Short-Form Era (2015-2025),100,9.0,91,2
Batman Begins,2005-06-15,Action,Pre Short-Form (2005-2014),100,11.846153846153847,91,2
Black Panther,2018-02-16,Action,Short-Form Era (2015-2025),100,6.670329670329671,91,1
Bohemian Rhapsody,2018-11-02,Drama,Short-Form Era (2015-2025),100,7.78021978021978,91,1
Bridesmaids,2011-05-13,Comedy,Pre Short-Form (2005-2014),100,10.780219780219781,91,1
Brokeback Mountain,2005-12-09,Drama,Pre Short-Form (2005-2014),100,12.87912087912088,91,1
Casino Royale,2006-11-17,Action,Pre Short-Form (2005-2014),100,9.516483516483516,91,1


era,count
Short-Form Era,8
Transition,3


hype_peak_score
100
100
100
100
100
100
100
100
100
100


genre,avg_hype_score
Horror,24.47252747252747
Drama,10.686813186813186
Action,9.83359497645212
Comedy,9.340659340659341


days_tracked
91
91
91
91
91
91
91
91
91
91
